<a href="https://colab.research.google.com/github/mnehan67/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mnehan67/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

**Lane:** CTR / Engagement Opportunity Scoring  
**This week:** March 2026 only (`month=2026-03`) — a mid-panel month, **not** the June `_sample`.

This notebook follows the assignment literally:
1. five plain-word contract answers;
2. **exactly three verification queries** (grain, count/date span, availability with `IS TRUE`);
3. **exactly five honest features**;
4. one deliberate label-derived leak, then deletion;
5. one named limitation.

The score below is a learning check, not the final capstone model.


In [1]:
# Install the small set of packages this notebook needs.
%pip -q install -U duckdb huggingface_hub scikit-learn pandas


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 75.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.9/842.9 kB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 112.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 108.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.6 which is incompatible.


In [2]:
import getpass
import duckdb
import numpy as np
import pandas as pd
from huggingface_hub import HfApi, hf_hub_download
from huggingface_hub.utils import HfHubHTTPError

DATASET_ID = "FlyRank/internship-warehouse"

# IMPORTANT:
# We intentionally IGNORE Colab Secrets / environment variables in this notebook
# so a stale or broken HF_TOKEN cannot keep being reused.
HF_TOKEN = getpass.getpass(
    "Paste a NEW Hugging Face plain READ token here (hidden; it will NOT be saved): "
).strip()

if not HF_TOKEN.startswith("hf_"):
    raise RuntimeError(
        "That does not look like a Hugging Face token. "
        "Create a token at Hugging Face Settings -> Access Tokens -> Create new token -> Read."
    )

# 1) Validate the token itself.
try:
    me = HfApi().whoami(token=HF_TOKEN)
    print(f"✓ Token valid for Hugging Face account: {me.get('name', 'unknown')}")
except Exception as e:
    raise RuntimeError(
        "This token is INVALID. Do not continue to the FlyRank query yet.\n"
        "Create a brand-new plain Read token, copy the full hf_... value, "
        "restart the runtime, and paste it into this hidden prompt."
    ) from e

# 2) Validate gated FlyRank dataset access with the SAME token.
try:
    _probe = hf_hub_download(
        repo_id=DATASET_ID,
        repo_type="dataset",
        filename="dim_clients.parquet",
        token=HF_TOKEN,
    )
    print("✓ Token can read the gated FlyRank internship dataset.")
except HfHubHTTPError as e:
    status = getattr(getattr(e, "response", None), "status_code", None)
    raise RuntimeError(
        f"The token itself is valid, but FlyRank dataset access failed (HTTP {status}).\n"
        "Open the FlyRank internship-warehouse dataset page in the SAME Hugging Face account, "
        "accept/request access, then create another plain Read token and retry."
    ) from e

# 3) Give DuckDB that exact validated token.
con = duckdb.connect()
con.execute("SET enable_progress_bar = false")
safe_token = HF_TOKEN.replace("'", "''")
con.execute(
    f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{safe_token}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"
MAR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

# Tiny real March access check.
march_rows = con.sql(f"SELECT COUNT(*) FROM {MAR}").fetchone()[0]

print("✓ DuckDB authentication works.")
print(f"✓ March 2026 partition reachable: {march_rows:,} rows.")
print("You can now run Query 1.")


Paste a NEW Hugging Face plain READ token here (hidden; it will NOT be saved): ··········
✓ Token valid for Hugging Face account: muhnehh


dim_clients.parquet: reconstructing file:   0%|          |  0.00B / 3.38kB            

dim_clients.parquet: downloading bytes:           |  0.00B            

✓ Token can read the gated FlyRank internship dataset.
✓ DuckDB authentication works.
✓ March 2026 partition reachable: 9,841,378 rows.
You can now run Query 1.


### Authentication note

This version **does not read `HF_TOKEN` from the Colab vault at all**. That is intentional.

Your traceback says:

> `Invalid user token. The token from Google Colab vault is invalid.`

So the problem occurs **before** the FlyRank dataset is queried. Paste a newly-created **plain Read** token only into the hidden prompt above. The token is kept in RAM for this runtime and is not written into the notebook.


## 1. Unit of analysis + time window

### My contract — five plain-word answers

1. **What one row means:** in my final lane frame, one row is **one pseudonymized content page for one client, aggregated over March 2026** (`client_hash_id × content_hash_id`). The raw warehouse table is daily, so I aggregate the day dimension away.
2. **Table(s):** I use only `fact_content_daily_performance`, specifically the partition `month=2026-03`. This keeps the first contract small and easy to audit.
3. **Time window:** **2026-03-01 through 2026-03-31**. I intentionally do not use the June `_sample`, because June is the final month of the panel.
4. **What I would predict/rank:** I estimate **expected CTR (percentage points)** from five non-click features, then rank pages by `expected CTR - observed CTR`. A positive gap is a **review proxy**, not promised future clicks.
5. **One deliberate exclusion:** **same-window clicks / CTR are excluded from the honest feature set**, because observed CTR is calculated from those clicks. Using them would hand the answer to the model.

**Eligibility for the small modeling frame:** GSC data must be available and the page must have at least **100 March impressions**. The 100-impression floor is a simple learning threshold to reduce tiny-denominator noise.


## 2. Fields: feature / label / context / excluded

| Bucket | Fields used here | Why |
|---|---|---|
| **Features (5 only)** | `log_impressions`, `avg_position`, `days_with_impressions`, `position_volatility`, `impression_spikiness` | Search exposure/position context that does not use clicks |
| **Label / proxy** | `observed_ctr_pp = 100 × March clicks / March impressions` | The measured CTR to estimate for a position-adjusted review proxy |
| **Context** | `client_hash_id`, `content_hash_id` | Join/group/split keys only; never model features |
| **Excluded** | March `gsc_clicks`, any same-window CTR copy | They directly create the label and would leak the answer |

**Important interpretation:** this week is an **observational expected-CTR comparison at the end of March**, not a future-CTR forecast. A later modeling notebook can move to past → future windows.


## 3. Verify it with exactly three small queries

The next three cells are the assignment's **three verification queries**. Their outputs must remain visible when this notebook is committed.


In [3]:
# VERIFICATION QUERY 1/3 — GRAIN
# Proves the raw source has no duplicate date×client×content keys,
# and the page-month aggregation has one row per client×content key.

q1 = con.sql(f"""
WITH source_dupes AS (
    SELECT report_date, client_hash_id, content_hash_id
    FROM {MAR}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
),
page_month AS (
    SELECT client_hash_id, content_hash_id
    FROM {MAR}
    WHERE gsc_data_available IS TRUE
    GROUP BY 1, 2
),
page_month_dupes AS (
    SELECT client_hash_id, content_hash_id
    FROM page_month
    GROUP BY 1, 2
    HAVING COUNT(*) > 1
)
SELECT
    (SELECT COUNT(*) FROM source_dupes)      AS raw_duplicate_groups,
    (SELECT COUNT(*) FROM page_month)        AS page_month_rows,
    (SELECT COUNT(*) FROM page_month_dupes)  AS page_month_duplicate_groups
""").df()

display(q1)
assert q1.loc[0, "raw_duplicate_groups"] == 0
assert q1.loc[0, "page_month_duplicate_groups"] == 0


,raw_duplicate_groups,page_month_rows,page_month_duplicate_groups
0,0,176738,0


In [4]:
# VERIFICATION QUERY 2/3 — ROW COUNT + DATE SPAN

q2 = con.sql(f"""
SELECT
    COUNT(*) AS rows_in_partition,
    MIN(report_date) AS first_day,
    MAX(report_date) AS last_day,
    COUNT(DISTINCT report_date) AS distinct_days,
    COUNT(DISTINCT client_hash_id) AS clients,
    COUNT(DISTINCT content_hash_id) AS content_items
FROM {MAR}
""").df()

display(q2)
assert str(q2.loc[0, "first_day"])[:10] == "2026-03-01"
assert str(q2.loc[0, "last_day"])[:10] == "2026-03-31"


,rows_in_partition,first_day,last_day,distinct_days,clients,content_items
0,9841378,2026-03-01,2026-03-31,31,55,331437


In [5]:
# VERIFICATION QUERY 3/3 — AVAILABILITY
# Requirement: explicitly filter with IS TRUE and show how many rows survive.

q3 = con.sql(f"""
SELECT
    (SELECT COUNT(*) FROM {MAR}) AS total_rows,
    (SELECT COUNT(*) FROM {MAR}
        WHERE gsc_data_available IS TRUE) AS rows_surviving_gsc_available,
    ROUND(
        100.0 *
        (SELECT COUNT(*) FROM {MAR} WHERE gsc_data_available IS TRUE) /
        NULLIF((SELECT COUNT(*) FROM {MAR}), 0),
        2
    ) AS pct_surviving
""").df()

display(q3)
assert q3.loc[0, "rows_surviving_gsc_available"] <= q3.loc[0, "total_rows"]


,total_rows,rows_surviving_gsc_available,pct_surviving
0,9841378,3611061,36.69


### Five features — and “available when?”

1. **`log_impressions`** — knowable at the decision moment because March impressions have already been observed by March 31; log scaling only compresses the long tail.
2. **`avg_position`** — knowable because it is computed only from March search-position measurements; it is impression-weighted using `gsc_sum_position / gsc_impressions`.
3. **`days_with_impressions`** — knowable because it only counts March days on which the page received at least one impression.
4. **`position_volatility`** — knowable because it is the spread of March daily average position values; it uses no clicks or future dates.
5. **`impression_spikiness`** — knowable because it is the largest March daily-impression count divided by total March impressions; it measures whether visibility came from one spike or was steadier.

No feature above is calculated from March clicks.


In [6]:
# Build the lane feature frame from the SAME March partition.
# This is not one of the three verification queries above; it is the feature-building step.

frame = con.sql(f"""
WITH agg AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_march,
        SUM(gsc_clicks) AS clicks_march,
        SUM(gsc_sum_position) AS sum_position_march,
        COUNT(*) FILTER (WHERE gsc_impressions > 0) AS days_with_impressions,
        STDDEV_POP(gsc_avg_position)
            FILTER (WHERE gsc_impressions > 0) AS position_volatility,
        MAX(gsc_impressions) AS max_daily_impressions
    FROM {MAR}
    WHERE gsc_data_available IS TRUE
    GROUP BY 1, 2
)
SELECT
    client_hash_id,
    content_hash_id,

    -- exactly five honest features
    LN(1 + impressions_march) AS log_impressions,
    sum_position_march / NULLIF(impressions_march, 0) AS avg_position,
    days_with_impressions,
    COALESCE(position_volatility, 0.0) AS position_volatility,
    max_daily_impressions / NULLIF(impressions_march, 0) AS impression_spikiness,

    -- label / proxy (NOT a feature)
    100.0 * clicks_march / NULLIF(impressions_march, 0) AS observed_ctr_pp,

    -- kept only for readable eligibility/output checks
    impressions_march
FROM agg
WHERE impressions_march >= 100
  AND sum_position_march > 0
""").df()

FEATURES = [
    "log_impressions",
    "avg_position",
    "days_with_impressions",
    "position_volatility",
    "impression_spikiness",
]
LABEL = "observed_ctr_pp"

assert len(FEATURES) == 5
assert LABEL not in FEATURES
assert not any("click" in c.lower() or "ctr" in c.lower() for c in FEATURES)
assert not frame.duplicated(["client_hash_id", "content_hash_id"]).any()

print(f"Eligible page-month rows: {len(frame):,}")
print(f"Clients represented: {frame['client_hash_id'].nunique():,}")
print("Exactly five honest features:", FEATURES)

display(frame[FEATURES + [LABEL]].head(10))


Eligible page-month rows: 101,441
Clients represented: 44
Exactly five honest features: ['log_impressions', 'avg_position', 'days_with_impressions', 'position_volatility', 'impression_spikiness']


,log_impressions,avg_position,days_with_impressions,position_volatility,impression_spikiness,observed_ctr_pp
0,9.291920,8.049866,31,1.354219,0.054936,0.202784
1,6.559615,5.863830,31,3.918809,0.070922,0.141844
2,7.649693,2.632206,31,1.855609,0.080515,0.047642
3,5.666427,10.489583,30,5.300735,0.065972,0.000000
4,8.170751,3.151344,31,0.723586,0.082885,0.707214
5,7.351800,5.247112,31,1.361807,0.067394,0.192555
6,7.611842,7.853538,31,3.783562,0.066799,0.000000
7,7.137278,6.887828,31,3.141284,0.095465,0.000000
8,6.586172,22.888122,31,10.177622,0.082873,0.138122
9,5.087596,5.055901,26,3.675071,0.130435,0.000000


### The trap: deliberately leak the label once

I will first fit a quick grouped-by-client linear regression with the **five honest features**.

Then I will add **one intentionally illegal feature**:

`leaky_ctr_copy_pp = observed_ctr_pp`

That column is literally derived from the label, so the score should jump toward perfect. I then delete it and keep/report the honest score. This is the notebook-02 leakage lesson performed on the real warehouse slice.


In [7]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

model_df = frame.dropna(subset=FEATURES + [LABEL]).copy()

splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(
    splitter.split(model_df[FEATURES], model_df[LABEL], groups=model_df["client_hash_id"])
)

train = model_df.iloc[train_idx].copy()
test = model_df.iloc[test_idx].copy()

# ---- HONEST model: five allowed features only ----
honest_model = LinearRegression()
honest_model.fit(train[FEATURES], train[LABEL])
honest_pred = honest_model.predict(test[FEATURES])

honest_mae = mean_absolute_error(test[LABEL], honest_pred)
honest_r2 = r2_score(test[LABEL], honest_pred)

# ---- DELIBERATE LEAK: one label-derived column ----
LEAK = "leaky_ctr_copy_pp"
train[LEAK] = train[LABEL]   # illegal on purpose
test[LEAK] = test[LABEL]     # illegal on purpose

leaky_features = FEATURES + [LEAK]
leaky_model = LinearRegression()
leaky_model.fit(train[leaky_features], train[LABEL])
leaky_pred = leaky_model.predict(test[leaky_features])

leaky_mae = mean_absolute_error(test[LABEL], leaky_pred)
leaky_r2 = r2_score(test[LABEL], leaky_pred)

results = pd.DataFrame({
    "version": ["HONEST: 5 features", "LEAKY: + label copy"],
    "MAE_ctr_pp": [honest_mae, leaky_mae],
    "R2": [honest_r2, leaky_r2],
})
display(results.round(6))

print(f"Train clients: {train['client_hash_id'].nunique()}")
print(f"Test clients:  {test['client_hash_id'].nunique()}")
print(f"Client overlap: {len(set(train.client_hash_id) & set(test.client_hash_id))}")
assert len(set(train.client_hash_id) & set(test.client_hash_id)) == 0

print("\nLeak lesson:")
print("The leaky score is near-perfect because the answer was put into the inputs.")
print("That is not model skill.")

# DELETE the leak and keep the honest feature list.
train.drop(columns=[LEAK], inplace=True)
test.drop(columns=[LEAK], inplace=True)
assert LEAK not in train.columns and LEAK not in test.columns
assert len(FEATURES) == 5

print("\nLeak deleted.")
print("FINAL feature list:", FEATURES)
print(f"FINAL honest MAE: {honest_mae:.4f} CTR percentage points")
print(f"FINAL honest R²:  {honest_r2:.4f}")


,version,MAE_ctr_pp,R2
0,HONEST: 5 features,0.345678,0.005909
1,LEAKY: + label copy,0.000000,1.000000


Train clients: 33
Test clients:  11
Client overlap: 0

Leak lesson:
The leaky score is near-perfect because the answer was put into the inputs.
That is not model skill.

Leak deleted.
FINAL feature list: ['log_impressions', 'avg_position', 'days_with_impressions', 'position_volatility', 'impression_spikiness']
FINAL honest MAE: 0.3457 CTR percentage points
FINAL honest R²:  0.0059


## 4. Data limit

### Named limitation — single-month observational proxy

This March-only slice can tell me which pages had CTR that was high or low **relative to patterns observed in March**, but it cannot tell me whether a page will improve after an edit or whether the pattern persists into later months. Search demand, SERP layout, seasonality, and client history differ over time, so this week's output is **decision-support**, not a causal or future-traffic claim.


## 5. Self-check

After **Runtime → Run all**, confirm the three query outputs and the honest/leaky metric table are visible, then save this exact file to your repo.

- [x] Five plain-word contract answers are present
- [x] Exactly three verification queries are present
- [x] Availability uses `IS TRUE`
- [x] Exactly five honest features are built
- [x] Every feature has an “available when?” line
- [x] One label-derived leak is added on purpose
- [x] The leak is deleted and the honest score is retained
- [x] One named limitation is stated
- [ ] **Runtime → Run all** completes with no errors
- [ ] Executed outputs are saved in `work/notebooks/w03_data_contract.ipynb`
- [ ] Commit to GitHub, then submit the repo URL

**Commit message suggestion:** `Complete W03 data contract and leakage check on March warehouse slice`
